In [1]:
# Install timm for models and ptflops for efficiency metrics
!pip install -q timm ptflops gdown

In [2]:
import torch
import timm
import os
from ptflops import get_model_complexity_info

print(f"PyTorch version: {torch.__version__}")
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# Quick test of your first model selection
model = timm.create_model('resnet50', pretrained=True)
print("ResNet50 loaded successfully!")

# Directory to save your plots, tables, and model checkpoints
os.makedirs('/kaggle/working/results/plots', exist_ok=True)
os.makedirs('/kaggle/working/results/checkpoints', exist_ok=True)

PyTorch version: 2.9.0+cu126
GPU Available: True
GPU Model: Tesla T4


model.safetensors:   0%|          | 0.00/102M [00:00<?, ?B/s]

ResNet50 loaded successfully!


In [3]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

# 1. Define standard transforms (ImageNet normalization is standard for timm models)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# 2. Load the full dataset (Assuming you downloaded/unzipped it to /kaggle/working/data/AID)
# Replace with your actual path
data_dir = '/kaggle/input/datasets/adarshguduru/gnr638-ass2/train_data' 
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

# 3. Stratified Split (80% Train, 20% Val)
# Note: For the assignment, use a fixed seed for reproducibility!
seed = 42 
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size], 
                                          generator=torch.Generator().manual_seed(seed))

# 4. DataLoaders (Optimized for 2 GPUs)
# Since you have 2 GPUs, you can double your batch size safely.
batch_size = 64 
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)

print(f"Classes: {len(full_dataset.classes)} | Total Images: {len(full_dataset)}")

Classes: 30 | Total Images: 6993


In [4]:
import timm
import torch.nn as nn

def get_model(model_name, num_classes=30):
    # Load pre-trained model
    model = timm.create_model(model_name, pretrained=True)
    
    # Identify and replace the head for each architecture
    if 'resnet' in model_name:
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif 'efficientnet' in model_name:
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif 'convnext' in model_name:
        model.head.fc = nn.Linear(model.head.fc.in_features, num_classes)
    
    # Move to GPU and wrap for 2x GPU usage
    model = model.cuda()
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
        
    return model

# Load your big three
model_resnet = get_model('resnet50')
model_efficient = get_model('efficientnet_b0')
model_convnext = get_model('convnext_tiny')

print("Models prepared for 30-class classification on 2x T4 GPUs.")

model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/114M [00:00<?, ?B/s]

Models prepared for 30-class classification on 2x T4 GPUs.


In [5]:
from ptflops import get_model_complexity_info

# Helper to report metrics
def report_efficiency(model, name):
    # Use the .module if it's wrapped in DataParallel
    target_model = model.module if isinstance(model, nn.DataParallel) else model
    
    macs, params = get_model_complexity_info(target_model, (3, 224, 224), 
                                            as_strings=True, 
                                            print_per_layer_stat=False)
    print(f"[{name}] MACs: {macs} | Params: {params}")

report_efficiency(model_resnet, "ResNet50")
report_efficiency(model_efficient, "EfficientNet-B0")
report_efficiency(model_convnext, "ConvNeXt-Tiny")

[ResNet50] MACs: 4.13 GMac | Params: 23.57 M
[EfficientNet-B0] MACs: 390.87 MMac | Params: 4.05 M
[ConvNeXt-Tiny] MACs: 4.48 GMac | Params: 27.84 M


In [6]:
from torch.utils.data import Subset
import numpy as np

def get_data_subset(full_dataset, percentage, seed=42):
    total_size = len(full_dataset)
    subset_size = int(percentage * total_size)
    
    # Create indices
    indices = np.arange(total_size)
    np.random.seed(seed)
    np.random.shuffle(indices)
    
    subset_indices = indices[:subset_size]
    return Subset(full_dataset, subset_indices)

In [7]:
import time
import json

def train_model(model, train_loader, val_loader, criterion, optimizer, num_epochs=30,model_name='experiment'):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    for epoch in range(num_epochs):
        start_time = time.time()
        
        # Training Phase
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        for images, labels in train_loader:
            images, labels = images.cuda(), labels.cuda()
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        train_acc = 100. * correct / total
        train_loss = running_loss / total

        # Validation Phase
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.cuda(), labels.cuda()
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * images.size(0)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += predicted.eq(labels).sum().item()
        
        val_acc = 100. * val_correct / val_total
        val_loss = val_loss / val_total
        
        # Save History
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        with open(f'/kaggle/working/{model_name}_history.json', 'w') as f:
            json.dump(history, f)
        
        duration = time.time() - start_time
        print(f"Epoch {epoch+1}/{num_epochs} | Train Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}% | Time: {duration:.1f}s")
        
    return history

In [8]:
data_regimes = [1.0, 0.20, 0.05] # 100%, 20%, 5%
models_to_test = ['resnet50', 'efficientnet_b0', 'convnext_tiny']

# Dictionary to store results for the final report
experiment_results = {}

for model_name in models_to_test:
    for regime in data_regimes:
        print(f"\n--- Starting Experiment: {model_name} with {int(regime*100)}% Data ---")
        
        # 1. Create subset
        # Note: We use the full_dataset defined earlier
        subset = get_data_subset(full_dataset, regime)
        loader = DataLoader(subset, batch_size=64, shuffle=True, num_workers=4, pin_memory=True)
        
        # 2. Get fresh model (fully unfrozen)
        model = get_model(model_name)

        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        
        # 3. Train
        # Use your train_model function here
        # Remember to use a unique model_name for checkpointing!
        run_name = f"{model_name}_{int(regime*100)}"
        history = train_model(model, loader, val_loader, criterion, optimizer, num_epochs=20, model_name=run_name)
        
        # 4. Store results
        experiment_results[run_name] = history

        del model
        del optimizer
        torch.cuda.empty_cache()


--- Starting Experiment: resnet50 with 100% Data ---
Epoch 1/20 | Train Acc: 23.45% | Val Acc: 51.11% | Time: 53.1s
Epoch 2/20 | Train Acc: 59.16% | Val Acc: 78.98% | Time: 46.8s
Epoch 3/20 | Train Acc: 80.75% | Val Acc: 88.92% | Time: 48.2s
Epoch 4/20 | Train Acc: 87.99% | Val Acc: 94.85% | Time: 47.6s
Epoch 5/20 | Train Acc: 92.09% | Val Acc: 96.85% | Time: 47.9s
Epoch 6/20 | Train Acc: 94.51% | Val Acc: 98.07% | Time: 47.5s
Epoch 7/20 | Train Acc: 96.17% | Val Acc: 98.86% | Time: 48.0s
Epoch 8/20 | Train Acc: 97.00% | Val Acc: 99.36% | Time: 48.6s
Epoch 9/20 | Train Acc: 97.88% | Val Acc: 99.71% | Time: 47.9s
Epoch 10/20 | Train Acc: 98.44% | Val Acc: 99.64% | Time: 48.1s
Epoch 11/20 | Train Acc: 98.73% | Val Acc: 99.93% | Time: 47.9s
Epoch 12/20 | Train Acc: 98.91% | Val Acc: 99.86% | Time: 47.6s
Epoch 13/20 | Train Acc: 99.30% | Val Acc: 99.93% | Time: 47.4s
Epoch 14/20 | Train Acc: 99.23% | Val Acc: 99.93% | Time: 48.0s
Epoch 15/20 | Train Acc: 99.37% | Val Acc: 99.93% | Time: 4

In [9]:
# Calculate Drop for each model
for model_name in models_to_test:
    acc_100 = experiment_results[f"{model_name}_100"]['val_acc'][-1]
    acc_5 = experiment_results[f"{model_name}_5"]['val_acc'][-1]
    
    delta = (acc_100 - acc_5) / acc_100
    print(f"[{model_name}] Performance Drop (Delta): {delta:.4f}")

[resnet50] Performance Drop (Delta): 0.6662
[efficientnet_b0] Performance Drop (Delta): 0.2275
[convnext_tiny] Performance Drop (Delta): 0.1537
